# AgriSense-MM: Interactive Walkthrough & Demonstration
**Course:** Neural Networks and Deep Learning (NNDL)  
**Team:** Sohum Chandra Srivastava (562), Anjul Shukla (572)  

This notebook demonstrates:
1. Loading the paired multimodal data (Canopy images, 14-day weather, 100 Hz IMU, Static GPS/Soil).
2. Loading the trained **AgriSenseMMNet** neural network checkpoint.
3. Running inference with **Monte Carlo (MC) Dropout** for epistemic uncertainty estimation.
4. Generating **Grad-CAM visual saliency maps** on leaf pathology.
5. Calculating **cross-attention modality attribution weights**.
6. Running the **Transportation Route Optimizer (2-Opt TSP)**.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from agrisense_mm.models import AgriSenseMMNet, GradCAMExplainer, overlay_gradcam_on_image, format_modality_attributions
from agrisense_mm.training.dataset import MultiModalCropDataset
from agrisense_mm.services.router import InspectionRouter
from agrisense_mm.data.constants import STRESS_CLASSES, RECOMMENDED_ACTIONS

### 1. Load Trained Checkpoint

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print('Using Device:', device)

model = AgriSenseMMNet().to(device)
chk = torch.load('checkpoints/best_agrisense_model.pt', map_location=device)
model.load_state_dict(chk['model_state_dict'])
model.eval()
print('Loaded checkpoint from epoch', chk.get('epoch', 1), 'with Val F1:', chk.get('val_f1', 0.0))

### 2. Multi-Task Inference & MC-Dropout Uncertainty

In [ ]:
ds = MultiModalCropDataset(data_dir='agrisense_mm/data/synthetic', split='val')
sample = ds[0]

img = sample['image'].unsqueeze(0).to(device)
env = sample['env_seq'].unsqueeze(0).to(device)
imu = sample['imu_window'].unsqueeze(0).to(device)
static = sample['static_feat'].unsqueeze(0).to(device)

mc_results = model.predict_uncertainty(img, env, imu, static, num_mc_samples=25)

pred_cls = mc_results['pred_class'].item()
pred_sev = mc_results['pred_severity'].item()
entropy = mc_results['predictive_entropy'].item()

print(f"Predicted Stress Class: {STRESS_CLASSES[pred_cls]}")
print(f"Predicted Severity:     {pred_sev:.1f} / 100")
print(f"Epistemic Uncertainty: {entropy:.3f} (0=certain, 1=uncertain)")
print(f"Recommended Action:    {RECOMMENDED_ACTIONS[pred_cls]}")

### 3. Grad-CAM Visual Explainability

In [ ]:
raw_img_path = os.path.join('agrisense_mm/data/synthetic/images', sample['image_file'])
raw_rgb = np.array(Image.open(raw_img_path).convert('RGB'))

explainer = GradCAMExplainer(model.visual_branch)
out = model(img, env, imu, static)
target_score = out['cls_logits'][0, pred_cls]
heatmap = explainer.generate_heatmap(target_score)
overlay = overlay_gradcam_on_image(raw_rgb, heatmap, alpha=0.55)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
ax1.imshow(raw_rgb)
ax1.set_title('Raw Foliage')
ax1.axis('off')

ax2.imshow(heatmap, cmap='jet')
ax2.set_title('Grad-CAM Saliency')
ax2.axis('off')

ax3.imshow(overlay)
ax3.set_title('Pathology Focus Overlay')
ax3.axis('off')
plt.tight_layout()
plt.show()

### 4. Transportation Route Optimization

In [ ]:
from agrisense_mm.services.storage import AgriSenseDatabase
db = AgriSenseDatabase('agrisense_mm.db')
zones = db.get_all_zones()

router = InspectionRouter()
route_plan = router.plan_smart_inspection_route(28.6139, 77.2090, zones)

print(f"Smart 2-Opt TSP Path: {route_plan['smart_route_distance_m']} m ({route_plan['smart_route_time_min']} min)")
print(f"Naive Zigzag Sweep:   {route_plan['naive_sweep_distance_m']} m ({route_plan['naive_sweep_time_min']} min)")
print(f"Distance Saved:       {route_plan['distance_saved_pct']}%")
print(f"Efficiency Gain:      {route_plan['efficiency_multiplier']}x faster")